In [12]:
from scipy.stats import norm
import math

# Premium Pricing A/B Test Design and Validation

## Overview

This project outlines the analytical framework for evaluating a pricing experiment on an online education platform. The goal is to assess whether lowering the price of a Premium subscription can improve user conversion enough to offset a potential decrease in revenue per paying user and ultimately increase overall revenue per user.

The analysis covers:

- experiment design
- sample size estimation
- expected test duration
- split system validation
- interpretation of statistical significance

## Business Context

The platform offers online courses and subscription-based access to premium content. A pricing experiment is being considered to measure the impact of a lower Premium subscription price on user purchase behavior.

The core business question is whether a lower entry price can drive a meaningful increase in conversion and improve top-line monetization efficiency.

## Product Hypothesis

Reducing the Premium subscription price will lead to a significant increase in purchase conversion rate. The gain in conversion is expected to compensate for a lower ARPPU and contribute to growth in overall ARPU.

### Expected impact

- Historical conversion rate: **0.5%**
- Expected conversion rate after the price change: **0.8%**

This implies a substantial relative uplift in purchase conversion and makes conversion rate the primary metric for experiment evaluation.

## Primary Metric

The target action is the **purchase** event, representing the completion of a course or subscription payment.

### Main metric

**Conversion Rate (CVR)**

Conversion Rate is defined as the share of users who complete the purchase action. It is the key metric used to evaluate whether the pricing change successfully increases monetization efficiency at the top of the funnel.

## Data Model

The experiment analysis relies on two layers of data:

### Historical pre-experiment data

These tables contain platform activity for the **7 days preceding the experiment** and are used for baseline estimation and traffic analysis.

#### `pre_courses`

| Column | Type |
|------|------|
| course_id | INT PRIMARY KEY |
| course_name | VARCHAR |
| total_modules | INT |

#### `pre_users`

| Column | Type |
|------|------|
| user_id | INT PRIMARY KEY |
| registration_date | DATE |
| region | VARCHAR |
| device_type | VARCHAR |

#### `pre_events`

| Column | Type |
|------|------|
| event_id | INT PRIMARY KEY |
| user_id | INT |
| course_id | INT |
| event_timestamp | DATETIME |
| event_type | VARCHAR |
| value | DECIMAL(10, 2) |

---

### Experiment data

These tables store user assignments and behavioral events collected during the live A/B test.

#### `courses`

| Column | Type |
|------|------|
| course_id | INT PRIMARY KEY |
| course_name | VARCHAR |
| total_modules | INT |

#### `users`

| Column | Type |
|------|------|
| user_id | INT PRIMARY KEY |
| registration_date | DATE |
| region | VARCHAR |
| device_type | VARCHAR |
| test_group | VARCHAR |

#### `events`

| Column | Type |
|------|------|
| event_id | INT PRIMARY KEY |
| user_id | INT |
| course_id | INT |
| event_timestamp | DATETIME |
| event_type | VARCHAR |
| value | DECIMAL(10, 2) |

## Experiment Assumptions

The following assumptions are used in experiment planning:

- Baseline conversion rate: **0.005**
- Expected conversion rate: **0.008**
- Significance level: **0.05**
- Statistical power: **0.8**
- Experiment split: **50/50**
- Minimum behavioral cycle length: **7 days**

The platform demonstrates a clear weekly usage pattern, so any experiment must run for at least one full weekly cycle, even if the target sample size is reached earlier.

## Analytical Objectives

The project is structured around the following questions:

1. **How many users are required per group** to detect the expected uplift in conversion?
2. **What is the average DAU of new users**, based on pre-experiment data?
3. **How long should the test run** to collect the required sample?
4. **Did the split system allocate users correctly** between control and treatment?
5. **Is the observed deviation from a 50/50 split statistically acceptable?**

## Statistical Approach

### Sample size estimation

The required sample size is calculated using a two-sided test for proportions based on:

- baseline CVR
- expected CVR uplift
- alpha = 0.05
- power = 0.8

### Test duration estimation

Test duration is estimated using:

- required total sample size across both groups
- average daily number of new users

### Split system validation

To validate the allocation logic, the observed share of users in the control group is compared with the expected 50% share using a one-sample proportion z-test.

This ensures that any imbalance between groups is consistent with random variation rather than an implementation issue.

## Practical Constraints

Even when traffic volume allows the required sample size to be collected in fewer days, the experiment should not be stopped prematurely. Because the product has a **7-day usage cycle**, a valid reading requires at least one full week of exposure to capture weekday and weekend behavior consistently.

## Expected Outcome

A successful experiment design should provide:

- statistically justified sample size
- realistic estimate of test duration
- validated split quality
- a reliable framework for interpreting conversion lift

This creates the foundation for making a confident product decision on Premium pricing.

In [4]:
# --- 1. Statistical parameters ---
alpha = 0.05      # Type I error rate
power = 0.8       # Statistical power

# Z critical values
Z_alpha_half = norm.ppf(1 - alpha / 2)  # Z(0.025) ≈ 1.96
Z_beta = norm.ppf(power)                # Z(0.80) ≈ 0.84

# --- 2. A/B test parameters ---
P1 = 0.005  # Historical CVR (baseline conversion rate)
P2 = 0.008  # Expected CVR after the change

# --- 3. Calculate the required sample size per group ---
# Formula:
# N_per_group = (Z_alpha/2 + Z_beta)^2 * (P1(1-P1) + P2(1-P2)) / (P2 - P1)^2

numerator_p = P1 * (1 - P1) + P2 * (1 - P2)
denominator_p = (P2 - P1) ** 2
Z_term = (Z_alpha_half + Z_beta) ** 2

N_per_group = (Z_term * numerator_p) / denominator_p

# --- 4. Print result ---
print(f"N (required per group): {N_per_group:.2f} users")

N (required per group): 11259.65 users


# 1 - (SQL) Calculate Average DAU of New Users

## Task Description

You have already calculated the required **sample size for one group** in an A/B test.
Now you need to estimate the **Daily Active Users (DAU) of new users** in order to calculate the **expected duration of the experiment**.

Use the `pre_users` table for this task.

The table `pre_users` contains data for **exactly 7 days prior to the start of the A/B test**.

Your goal is to calculate the **average DAU of unique new users** over this 7-day period.


## Table Structure

### `pre_users`

| Column | Type | Description |
|------|------|------|
| user_id | INT | Unique user identifier |
| registration_date | DATE | Date when the user registered |
| region | VARCHAR | User region |
| device_type | VARCHAR | Device type used during registration |

## Task

1. Calculate the number of **unique users registered per day**.
2. Compute the **average DAU of new users** across the 7-day period.

In [5]:
# SELECT AVG(daily_users) AS avg_dau_new_users
# FROM (
#     SELECT
#         registration_date,
#         COUNT(DISTINCT user_id) AS daily_users
#     FROM pre_users
#     GROUP BY registration_date
# ) t;

## SQL Query Result

The SQL query returned the following result:

| avg_dau_new_users |
|-------------------|
| 4333.4286 |

This means that the **average number of new users per day (DAU)** during the 7-day pre-test period is approximately **4,333 users**.

# 2 - Calculate the Duration of the A/B Test

You already calculated:

- the **required sample size per group**
- the **average DAU of new users**

Now you need to estimate the **duration of the experiment in days**.

## Known values

Sample size per group:

11259.65 users

Average DAU of new users:

4333.4286 users per day

## Task

Calculate the expected **duration of the A/B test in days** using the formula:


In [10]:
sample_size_per_group = 11259.65
avg_dau_new_users = 4333.4286
number_of_groups = 2

test_duration_days = (sample_size_per_group * number_of_groups) / avg_dau_new_users

print(f"Estimated experiment duration: {test_duration_days:.2f} days")

Estimated experiment duration: 5.20 days


In [11]:
sample_size_per_group = 11259.65
avg_dau_new_users = 4333.4286
number_of_groups = 2

estimated_days = (sample_size_per_group * number_of_groups) / avg_dau_new_users
final_duration = max(7, estimated_days)

print(f"Estimated duration based on traffic: {estimated_days:.2f} days")
print(f"Final test duration: {final_duration:.0f} days")

Estimated duration based on traffic: 5.20 days
Final test duration: 7 days


# 3 - A/B Test Validation: Split System Check

## Task Description

The A/B test has been successfully launched and completed.
The first step in analyzing the results is to verify that the **split system worked correctly**.

The A/B testing system collected all experiment data.
Assume that the database contains **only the relevant data for this experiment**.

Ideally, users should be distributed between the **control** and **test** groups in a **50/50 ratio**.
However, in practice, small deviations may occur due to randomness.

Your task is to verify that the observed deviation is acceptable by calculating the **distribution of users across the groups**.

## Database Schema

### `courses`

| Column | Type |
|------|------|
| course_id | INT PRIMARY KEY |
| course_name | VARCHAR |
| total_modules | INT |

### `users`

| Column | Type |
|------|------|
| user_id | INT PRIMARY KEY |
| registration_date | DATE |
| region | VARCHAR |
| device_type | VARCHAR |
| test_group | VARCHAR |

### `events`

| Column | Type |
|------|------|
| event_id | INT PRIMARY KEY |
| user_id | INT (FOREIGN KEY → users.user_id) |
| course_id | INT (FOREIGN KEY → courses.course_id) |
| event_timestamp | DATETIME |
| event_type | VARCHAR |
| value | DECIMAL(10,2) |

## Task

Calculate the results of the **split system distribution**.

For each test group, output:

- the **group name**
- the **total number of users in the group**
- the **percentage of users in the group**

Sort the results **alphabetically by test group**.

## Expected Output Columns

| Column | Description |
|------|------|
| `test_group` | Control or test group |
| `total_users_in_group` | Number of users assigned to the group |
| `group_share_percent` | Percentage of users in the group |

```sql
SELECT
    test_group,
    COUNT(user_id) AS total_users_in_group,
    COUNT(user_id) * 100.0 / SUM(COUNT(user_id)) OVER () AS group_share_percent
FROM users
GROUP BY test_group
ORDER BY test_group;
```

## Query Result

The SQL query returned the following distribution of users between the experimental groups:

| test_group | total_users_in_group | group_share_percent |
|-------------|----------------------|---------------------|
| control | 15302 | 50.44505 |
| test | 15032 | 49.55495 |

## Interpretation

The users were distributed almost evenly between the two groups:

- **Control group:** 50.45%
- **Test group:** 49.55%

The deviation from the ideal **50/50 split** is very small and is within the range of expected random variation.

## Conclusion

The split system appears to be functioning correctly.
The user allocation between the control and test groups is sufficiently balanced, meaning the experiment results can be analyzed without concerns about allocation bias.

# 4 - Split System Evaluation

## Task Description

Evaluate the performance of the split system and determine whether it is working correctly.

Use the following values in the Python template:

- the number of users assigned to the **control group**
- the **total number of users**
- the **target allocation share** between groups

Use **Python 3.10** for all calculations.

## Objective

In an ideal A/B test setup, users should be distributed between the control and test groups according to the target proportion. In this case, the expected allocation is **50/50**.

Your task is to check whether the observed deviation from the expected share is statistically significant.

## Statistical Check

Use a **two-sided one-sample proportion z-test** to compare:

- the **observed proportion** of users in the control group
- the **expected proportion** of **0.5**

## What to Include

In the code template, fill in:

- `success_control` — the number of users in the control group
- `n_total` — the total number of users
- `p_expected` — the expected proportion of users in the control group

Then calculate the **p-value** and interpret the result.

Also include in your comments:

- the **p-value** you obtained
- a brief explanation of how it should be interpreted

## Expected Output

```text
P-value > 0.05. The split system works correctly (the deviation is not statistically significant).

In [15]:
# --- 1. Input data for the split system test ---
# K = number of 'successes' (users assigned to the Control group)
success_control = 15302

# N = total number of observations (all users)
n_total = 30334

# Expected allocation proportion
p_expected = 0.5

# --- 2. Calculate Z-statistic for one proportion ---

# Observed proportion
p_observed = success_control / n_total

# Standard error for one proportion
se = math.sqrt(p_expected * (1 - p_expected) / n_total)

# Z-statistic
z_stat = (p_observed - p_expected) / se

# --- 3. Calculate p-value (two-sided test) ---
p_value = norm.sf(abs(z_stat)) * 2

# Interpretation
if p_value < 0.05:
    print("P-value < 0.05. The split system may not be working correctly.")
else:
    print("P-value > 0.05. The split system works correctly (the deviation is not statistically significant).")

P-value > 0.05. The split system works correctly (the deviation is not statistically significant).


# 5 - Conversion Rate Analysis for A/B Test Groups

## Task Description

After validating that the split system works correctly, the next step is to evaluate how the experiment affected user conversion.

Using the available experiment data, calculate the **conversion rate (CVR)** for both the **control** and **test** groups.

The conversion is defined as the proportion of users who performed the **purchase** event.

## Database Schema

### `courses`

| Column | Type |
|------|------|
| course_id | INT PRIMARY KEY |
| course_name | VARCHAR |
| total_modules | INT |

### `users`

| Column | Type |
|------|------|
| user_id | INT PRIMARY KEY |
| registration_date | DATE |
| region | VARCHAR |
| device_type | VARCHAR |
| test_group | VARCHAR |

### `events`

| Column | Type |
|------|------|
| event_id | INT PRIMARY KEY |
| user_id | INT |
| course_id | INT |
| event_timestamp | DATETIME |
| event_type | VARCHAR |
| value | DECIMAL(10,2) |

## Task

Calculate the conversion metrics for each test group.

For each group determine:

- the total number of users
- the number of unique users who made a purchase
- the conversion rate (CVR) expressed **in percent**

Sort the result **alphabetically by test group**.

## Expected Output Columns

| Column | Description |
|------|------|
| `test_group` | Control or test group |
| `total_users` | Total number of users in the group |
| `unique_purchasers` | Number of users who completed a purchase |
| `cvr` | Conversion rate in percent |

```sql

SELECT
    u.test_group,
    COUNT(DISTINCT u.user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN e.event_type = 'purchase' THEN u.user_id END) AS unique_purchasers,
    COUNT(DISTINCT CASE WHEN e.event_type = 'purchase' THEN u.user_id END) * 100.0
        / COUNT(DISTINCT u.user_id) AS cvr
FROM users u
LEFT JOIN events e
    ON u.user_id = e.user_id
GROUP BY u.test_group
ORDER BY u.test_group;
```

## Conversion Results Interpretation

The calculated conversion rates for the experiment groups are:

| Group | Total Users | Purchasers | CVR (%) |
|------|-------------|------------|---------|
| Control | 15,120 | 74 | 0.489 |
| Test | 15,214 | 163 | 1.071 |

### Key Observations

The **test group shows a significantly higher conversion rate** compared to the control group.

- Control group CVR: **0.49%**
- Test group CVR: **1.07%**

This represents an **absolute increase of approximately 0.58 percentage points**.

In relative terms, the conversion rate in the test group is more than **2 times higher** than in the control group.

### Preliminary Conclusion

The pricing change (or experiment treatment) appears to have a **strong positive impact on conversion**. Users exposed to the test variant were significantly more likely to complete a purchase.

However, before making a final product decision, it is necessary to perform a **statistical significance test** (for example, a two-proportion z-test) to verify that this difference is not caused by random variation.

# 6 - Statistical Significance Test for Conversion Rate (CVR)

## Task Description

After calculating the conversion rates for the control and test groups, the next step is to evaluate whether the observed difference in conversion is statistically significant.

Use the results obtained in the previous step and insert them into the provided Python template.

## Input Data

You need to provide the following values:

- Number of users who completed the target action in the **control group**
- Number of users who completed the target action in the **test group**
- Total number of users in the **control group**
- Total number of users in the **test group**

Use **Python 3.10** to perform the calculations.

## Statistical Method

Apply a **Z-test for two proportions** to determine whether the conversion rate in the test group is significantly higher than in the control group.

The test compares:

- **CVR_control = purchases_control / users_control**
- **CVR_test = purchases_test / users_test**

The test is performed as a **one-sided hypothesis test**:

- **H₀ (null hypothesis):** The conversion rate in the test group is not greater than in the control group.
- **H₁ (alternative hypothesis):** The conversion rate in the test group is greater than in the control group.

## Discussion Questions

Please answer the following in your comments:

1. What **p-value** did you obtain? How can it be interpreted?
2. How were users distributed when validating the **split system** and when calculating **CVR**?
   Should these numbers differ in a real experiment?

In [19]:
import math
from scipy.stats import norm

# --- 1. Input data ---
# Number of 'successes' (purchasers)
k_control = 74
k_test = 163

# Total number of observations (users)
n_control = 15120
n_test = 15214

# --- 2. Calculate p-value (Z-test for two proportions) ---

# Observed proportions (CVR)
p_control = k_control / n_control
p_test = k_test / n_test

# Pooled proportion
p_pool = (k_control + k_test) / (n_control + n_test)

# Standard error of the difference
# SE = sqrt( P_pool * (1 - P_pool) * (1/n1 + 1/n2) )
se = math.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_test))

# Z-statistic
z_stat = (p_test - p_control) / se

# --- 3. Calculate p-value (one-sided test: Test > Control) ---
p_value = 1 - norm.cdf(z_stat)

# --- 4. Output the result ---
print("--- Z-test Results for CVR ---")

if p_value < 0.05:
    print("P-value < 0.05. Statistical significance detected!")
else:
    print("P-value > 0.05. No statistical significance.")

--- Z-test Results for CVR ---
P-value < 0.05. Statistical significance detected!


# 7 - ARPU Analysis for Control and Test Groups

## Task Description

Conversion alone is not enough to evaluate the success of an experiment if the change leads to revenue loss.

The next step is to analyze how the experiment affected **ARPU (Average Revenue Per User)**.

Using the experiment data, calculate ARPU for both the **control** and **test** groups.

## Database Schema

### `courses`

| Column | Type |
|------|------|
| course_id | INT PRIMARY KEY |
| course_name | VARCHAR |
| total_modules | INT |

### `users`

| Column | Type |
|------|------|
| user_id | INT PRIMARY KEY |
| registration_date | DATE |
| region | VARCHAR |
| device_type | VARCHAR |
| test_group | VARCHAR |

### `events`

| Column | Type |
|------|------|
| event_id | INT PRIMARY KEY |
| user_id | INT |
| course_id | INT |
| event_timestamp | DATETIME |
| event_type | VARCHAR |
| value | DECIMAL(10, 2) |

## Task

Calculate **ARPU** for the **control** and **test** groups.

For each group, return:

- the group name
- the total number of users
- the total revenue
- ARPU

Sort the result **alphabetically by group**.

## Expected Output Columns

| Column | Description |
|------|------|
| `test_group` | Control or test group |
| `total_users` | Total number of users in the group |
| `total_revenue` | Total revenue generated by the group |
| `arpu` | Average revenue per user |

```sql
SELECT
    u.test_group,
    COUNT(DISTINCT u.user_id) AS total_users,
    COALESCE(SUM(CASE WHEN e.event_type = 'purchase' THEN e.value ELSE 0 END), 0) AS total_revenue,
    COALESCE(SUM(CASE WHEN e.event_type = 'purchase' THEN e.value ELSE 0 END), 0)
        / COUNT(DISTINCT u.user_id) AS arpu
FROM users u
LEFT JOIN events e
    ON u.user_id = e.user_id
GROUP BY u.test_group
ORDER BY u.test_group;
```

## ARPU Analysis Interpretation

The calculated revenue metrics for the experiment groups are:

| Group | Total Users | Total Revenue | ARPU |
|------|-------------|--------------|------|
| Control | 15,136 | 636,361.52 | 42.04 |
| Test | 15,198 | 830,833.89 | 54.67 |

### Key Observations

The **test group demonstrates significantly higher revenue performance** compared to the control group.

- **Control group ARPU:** 42.04
- **Test group ARPU:** 54.67

This represents an **absolute increase of approximately 12.63 in ARPU per user**.

In relative terms, ARPU in the test group increased by approximately **30% compared to the control group**.

### Interpretation

The experimental change not only increased the conversion rate observed earlier but also resulted in **higher average revenue per user**. This indicates that the experiment positively affected both **user monetization efficiency** and **overall revenue generation**.

### Preliminary Conclusion

The experiment appears to have a **strong positive financial impact**. The test group generates higher revenue and higher ARPU than the control group.

However, to make a final product decision, it is still necessary to verify whether the ARPU difference is **statistically significant**, typically using a **t-test or bootstrap analysis**.

# 8 - ARPPU Analysis for Correct ARPU Variance Estimation

## Task Description

ARPU is a tricky metric because its distribution contains two fundamentally different user groups:

- the majority of users who **do not pay** and therefore generate **zero revenue**
- a minority of users who **do make purchases** and generate **positive revenue**

Because of this, ARPU consists of a **large number of zeros and a small number of relatively high values**.
This structure makes it difficult to directly estimate the variance of ARPU using the standard variance formula.

To correctly estimate the variance of ARPU for statistical testing, the metric can be decomposed as:

$ARPU = ARPPU \times CVR$

Where:

- **ARPU** – Average Revenue Per User
- **ARPPU** – Average Revenue Per Paying User
- **CVR** – Conversion Rate

```sql
SELECT
    u.test_group,
    ROUND(AVG(e.value), 2) AS arppu_mean,
    ROUND(STDDEV_POP(e.value), 2) AS stddev_arppu_sp
FROM users u
JOIN events e
    ON u.user_id = e.user_id
WHERE e.event_type = 'purchase'
GROUP BY u.test_group
ORDER BY u.test_group;
```

## 9 - ARPPU Analysis Interpretation

The calculated metrics for paying users in each group are:

| Group | ARPPU Mean | ARPPU Std Dev |
|------|------------|---------------|
| Control | 8829.61 | 2064.01 |
| Test | 4939.36 | 1154.16 |

### Key Observations

1. **Average Revenue Per Paying User (ARPPU) decreased significantly in the test group.**

- Control ARPPU: **8829.61**
- Test ARPPU: **4939.36**

This represents a decrease of approximately **44%** in the average amount spent by paying users.

2. **Revenue variability also decreased in the test group.**

- Control standard deviation: **2064.01**
- Test standard deviation: **1154.16**

This suggests that spending behavior in the test group became **more concentrated around lower purchase values**, which is consistent with a **reduced price scenario**.

### Interpretation

The experiment likely involved **lowering the price of the premium product or subscription**. As expected, this caused users who completed a purchase to **spend less on average**.

However, this result should **not be evaluated in isolation**, because overall revenue per user depends on both:

ARPU = ARPPU × CVR

Where:
- **ARPPU** reflects how much paying users spend
- **CVR** reflects how many users convert to paying users

Earlier results showed that the **conversion rate increased significantly in the test group**, meaning more users completed purchases.

### Business Implication

The test created a classic trade-off:

- **Lower revenue per paying user**
- **Higher conversion rate**

The final decision about the experiment should therefore be based on **ARPU**, which captures the combined effect of both changes. If ARPU increased, the pricing change can be considered **economically beneficial** despite the decrease in ARPPU.

## Statistical Significance Test for ARPU

To evaluate the economic impact of the experiment, we need to test whether the difference in **ARPU (Average Revenue Per User)** between the control and test groups is statistically significant.

Unlike conversion rate, ARPU has a **mixed distribution**, because it consists of two types of users:

- the majority of users generate **zero revenue** (non-paying users)
- a smaller group of users generates **positive revenue** (paying users)

Because of this, the standard variance formula cannot be applied directly.
Instead, ARPU can be decomposed as:

$
ARPU = ARPPU \times CVR
$

Where:

- **ARPU** – Average Revenue Per User
- **ARPPU** – Average Revenue Per Paying User
- **CVR** – Conversion Rate

Using this decomposition, the variance of ARPU can be estimated as:

$
Var(ARPU) \approx (s_p^2 \cdot p) + (\bar{x}_p^2 \cdot p \cdot (1 - p))
$

Where:

- \(p\) — conversion rate (CVR)
- \(\bar{x}_p\) — mean ARPPU
- \(s_p\) — standard deviation of ARPPU

### Task

Using the results from the previous steps, perform a **Z-test for ARPU**.

Insert the following values into the template:

- Mean ARPU for the **control group**
- Mean ARPU for the **test group**
- Total number of users in the **control group**
- Total number of users in the **test group**
- Number of purchasers in the **control group**
- Number of purchasers in the **test group**
- Mean ARPPU for the **control group**
- Mean ARPPU for the **test group**
- Standard deviation of ARPPU for the **control group**
- Standard deviation of ARPPU for the **test group**


In [22]:
# --- 1. Input data ---

# Mean ARPU values
mean_control = 42.042912
mean_test = 54.667317

# Total number of users (N)
n_control = 15136
n_test = 15198

# Number of purchasers (K)
k_control = 74
k_test = 163

# Mean ARPPU values
mean_p_control = 8829.61
mean_p_test = 4939.36

# Standard deviation of ARPPU
std_p_control = 2064.01
std_p_test = 1154.16


# --- 2. ARPU variance calculation ---

# Conversion rates (CVR)
p_control = k_control / n_control
p_test = k_test / n_test

# ARPU variance formula
var_arpu_control = (std_p_control**2 * p_control) + (mean_p_control**2 * p_control * (1 - p_control))
var_arpu_test = (std_p_test**2 * p_test) + (mean_p_test**2 * p_test * (1 - p_test))


# --- 3. Z-statistic calculation ---

mean_diff = mean_test - mean_control

se = math.sqrt(
    (var_arpu_control / n_control) +
    (var_arpu_test / n_test)
)

z_stat = mean_diff / se


# --- 4. P-value calculation (one-sided test: Test > Control) ---

p_value = 1 - norm.cdf(z_stat)


# --- 5. Output results ---

print("--- Z-test Results for ARPU ---")

if p_value < 0.05:
    print("P-value < 0.05. The result is statistically significant.")
else:
    print("P-value > 0.05. No statistical significance.")

--- Z-test Results for ARPU ---
P-value < 0.05. The result is statistically significant.


# Final Experiment Results

The A/B test was conducted to evaluate the impact of a new pricing policy on user behavior and revenue metrics.

### Experiment Validation

- The **split system worked correctly**, and users were distributed between groups as expected.
- The **test duration was sufficient**, covering the full product usage cycle.

### Metric Results

- **Conversion Rate (CVR)** showed a **statistically significant increase** in the test group.
- **ARPPU (Average Revenue Per Paying User)** showed a **statistically significant decrease**.
- **ARPU (Average Revenue Per User)** showed a **statistically significant increase**.

### Interpretation

The pricing change reduced the amount paid by each individual paying user, which explains the decrease in ARPPU. However, the lower price significantly increased the number of users who completed a purchase, leading to a higher conversion rate.

Because the increase in conversion outweighed the decrease in ARPPU, the overall **ARPU increased**, indicating improved monetization efficiency.

### Business Recommendation

Since the experiment produced a **statistically significant increase in ARPU**, the new pricing policy **should be rolled out to the entire user base**.

### Notes on the Analysis

Small variations in the numerical results are possible because the dataset contains random components.

The analysis included the following steps:

- **DAU Calculation** – estimated the average number of daily new users based on historical data before the experiment.
- **Split System Validation** – verified that users were distributed evenly between control and test groups.
- **CVR Calculation** – measured the conversion rate (share of users who completed a purchase) in each group.
- **ARPU Calculation** – calculated the average revenue per user to evaluate the overall financial impact of the experiment.
- **ARPPU Calculation** – measured the average revenue per paying user to understand how the pricing change affected spending behavior among purchasers.

These metrics together allow us to evaluate both **user behavior changes** (conversion) and **monetization effects** (revenue per user), providing a complete picture of the experiment’s impact.